[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ELTE-DSED/Intro-Data-Security/blob/main/module_07_synthetic_data_generation/Lab_7_Tabular_Synthetic_Data_Generation.ipynb)

# **Lab 7a: Tabular Synthetic Data Generation**

**Course:** Introduction to Data Security Pr.  
**Module 7:**  Synthetic Data Generation  
**Estimated Time:** 90 minutes

---

By the end of this lab, you will understand:

1. **Synthetic Data:** Generating realistic but artificial data from distributions
2. **Privacy Benefits:** Synthetic data as privacy-preserving alternative to real data
3. **Generation Methods:** VAE, GAN, and diffusion-based approaches
4. **Utility Metrics:** Assessing synthetic data quality and realism
5. **Privacy Guarantees:** Formal differential privacy for synthetic data
6. **Real-World Applications:** Healthcare, finance, and tabular ML datasets

## Table of Contents

1. [Threat Model & Use Cases](#threat-model)
2. [VAE-Based Generation](#vae)
3. [GAN-Based Generation](#gan)
4. [Utility Evaluation](#utility)
5. [Privacy Analysis](#privacy)
6. [Exercises](#exercises)

## Threat Model & Use Cases <a id="threat-model"></a>

**Core Problem:** Share ML datasets publicly without revealing sensitive information

<div align=center>

| Domain | Real Data Risk | Synthetic Solution | Benefit |
|--------|----------------|--------------------|--------|
| **Healthcare** | Patient records, diagnoses | Synthetic patients with same disease distribution | HIPAA compliance |
| **Finance** | Transaction history, credit scores | Synthetic users with same spending patterns | PCI compliance |
| **Census** | Demographic information | Synthetic population with same statistics | GDPR compliance |

</div>

### Key Advantages:

- **Perfect Privacy:** No real individuals identifiable from synthetic data
- **Shareable:** Can publish synthetic data openly for research
- **Compliance:** Satisfies privacy regulations (GDPR, HIPAA, CCPA)
- **Utility:** Preserves statistical properties for ML model training

## Synthetic Data Generation Methods <a id="theory"></a>

### Method 1: Variational Autoencoder (VAE)

**Architecture:** Encoder → Latent Space → Decoder

- Learns continuous latent representation of data
- Sample from latent space → Generate new samples

### Method 2: Generative Adversarial Network (GAN)

**Architecture:** Generator (fake data) vs Discriminator (real/fake classifier)

- Generator learns to fool discriminator

### Method 3: Diffusion Models (Noise-Based)

**Architecture:** Iteratively add noise → Learn reverse process

- Denoise corrupted data back to real distribution

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Load and standardize Iris dataset

In [ ]:
iris = load_iris()
X_real = iris.data
y_real = iris.target

# Standardize
scaler = StandardScaler()
X_real_scaled = scaler.fit_transform(X_real)

# Create PyTorch dataset including labels
train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_real_scaled), torch.LongTensor(y_real)), 
    batch_size=32, 
    shuffle=True
)

print(f"Real data shape: {X_real_scaled.shape}")
print(f"Features: {iris.feature_names}")
print(f"Classes: {iris.target_names}")

## Privacy Utilities

We define the `sanitize_synthetic_data` function. It acts as a privacy guard by rejecting synthetic samples that are too close to any real training record, thereby mitigating Membership Inference risks.

In [ ]:
def sanitize_synthetic_data(synthetic_data_scaled, real_data_scaled, threshold=0.3):
    """
    Privacy Guard: Rejects synthetic samples that are within 'threshold' Euclidean distance 
    of any real training record. Helps pass Membership Inference Analysis.
    """
    dists = pairwise_distances(synthetic_data_scaled, real_data_scaled, metric='euclidean')
    min_dists = dists.min(axis=1)
    safe_indices = np.where(min_dists > threshold)[0]
    return safe_indices

## PART 1: Conditional VAE (CVAE)

In [ ]:
class CVAE(nn.Module):
    def __init__(self, input_dim=4, latent_dim=2, n_classes=3):
        super(CVAE, self).__init__()
        self.latent_dim = latent_dim
        self.label_emb = nn.Embedding(n_classes, n_classes)
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim + n_classes, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 8),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(8, latent_dim)
        self.fc_logvar = nn.Linear(8, latent_dim)
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim + n_classes, 8),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim)
        )
    
    def encode(self, x, labels):
        c = self.label_emb(labels)
        h = self.encoder(torch.cat([x, c], dim=1))
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z, labels):
        c = self.label_emb(labels)
        return self.decoder(torch.cat([z, c], dim=1))
    
    def forward(self, x, labels):
        mu, logvar = self.encode(x, labels)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, labels), mu, logvar, z
    
    def generate(self, n_samples, labels):
        with torch.no_grad():
            z = torch.randn(n_samples, self.latent_dim).to(device)
            samples = self.decode(z, labels).cpu().numpy()
        return samples


In [ ]:
def vae_loss(recon, x, mu, logvar, beta=15.0):
    recon_loss = nn.functional.mse_loss(recon, x, reduction='sum')
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return (recon_loss + beta * kld) / x.size(0)

def train_vae(model, train_loader, epochs=100, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            recon, mu, logvar, _ = model(x, y)
            loss = vae_loss(recon, x, mu, logvar)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        if (epoch + 1) % 20 == 0:
            print(f'Epoch {epoch+1}/{epochs}: Loss = {epoch_loss / len(train_loader):.4f}')
    return model

In [ ]:
print('Training CVAE...')
vae = CVAE(input_dim=4, latent_dim=2).to(device)
vae = train_vae(vae, train_loader, epochs=100)

In [ ]:
print('\nGenerating synthetic data with CVAE...')
n_oversample = 450
tiled_labels = np.tile(y_real, (n_oversample // len(y_real)) + 1)[:n_oversample]
synth_labels = torch.LongTensor(tiled_labels).to(device)

X_synthetic_vae_scaled = vae.generate(n_samples=n_oversample, labels=synth_labels)

# Apply Privacy Guard (threshold = 0.3)
safe_idx_vae = sanitize_synthetic_data(X_synthetic_vae_scaled, X_real_scaled, threshold=0.3)
X_synthetic_vae_scaled = X_synthetic_vae_scaled[safe_idx_vae]
y_synthetic_vae = tiled_labels[safe_idx_vae]

X_synthetic_vae = scaler.inverse_transform(X_synthetic_vae_scaled)

print(f'Retained {len(X_synthetic_vae)} samples after privacy filtering.')

## PART 2: Conditional GAN (CGAN)

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=10, output_dim=4, n_classes=3):
        super(Generator, self).__init__()
        self.label_emb = nn.Embedding(n_classes, n_classes)
        self.model = nn.Sequential(
            nn.Linear(latent_dim + n_classes, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, output_dim)
        )
    def forward(self, z, labels):
        c = self.label_emb(labels)
        return self.model(torch.cat([z, c], dim=1))

class Discriminator(nn.Module):
    def __init__(self, input_dim=4, n_classes=3):
        super(Discriminator, self).__init__()
        self.label_emb = nn.Embedding(n_classes, n_classes)
        self.model = nn.Sequential(
            nn.Linear(input_dim + n_classes, 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 16),
            nn.LeakyReLU(0.2),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    def forward(self, x, labels):
        c = self.label_emb(labels)
        return self.model(torch.cat([x, c], dim=1))

In [ ]:
def train_gan(generator, discriminator, train_loader, epochs=400):
    g_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
    d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))
    criterion = nn.BCELoss()
    
    for epoch in range(epochs):
        for x, y in train_loader:
            batch_size = x.size(0)
            x, y = x.to(device), y.to(device)
            
            # Train Discriminator
            d_optimizer.zero_grad()
            real_labels = torch.ones(batch_size, 1).to(device)
            fake_labels = torch.zeros(batch_size, 1).to(device)
            
            outputs = discriminator(x, y)
            d_loss_real = criterion(outputs, real_labels)
            
            z = torch.randn(batch_size, 10).to(device)
            fake_x = generator(z, y)
            outputs = discriminator(fake_x.detach(), y)
            d_loss_fake = criterion(outputs, fake_labels)
            
            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            d_optimizer.step()
            
            # Train Generator
            g_optimizer.zero_grad()
            outputs = discriminator(fake_x, y)
            g_loss = criterion(outputs, real_labels)
            g_loss.backward()
            g_optimizer.step()
            
        if (epoch + 1) % 100 == 0:
            print(f'Epoch {epoch+1}/{epochs}: D_loss={d_loss.item():.4f}, G_loss={g_loss.item():.4f}')

In [ ]:
print('Training CGAN...')
generator = Generator().to(device)
discriminator = Discriminator().to(device)
train_gan(generator, discriminator, train_loader, epochs=400)

In [ ]:
print('\nGenerating synthetic data with CGAN...')
with torch.no_grad():
    z = torch.randn(n_oversample, 10).to(device)
    X_synthetic_gan_scaled = generator(z, synth_labels).cpu().numpy()

# Apply Privacy Guard (threshold = 0.3)
safe_idx_gan = sanitize_synthetic_data(X_synthetic_gan_scaled, X_real_scaled, threshold=0.3)
X_synthetic_gan_scaled = X_synthetic_gan_scaled[safe_idx_gan]
y_synthetic_gan = tiled_labels[safe_idx_gan]

X_synthetic_gan = scaler.inverse_transform(X_synthetic_gan_scaled)

print(f'Retained {len(X_synthetic_gan)} samples after privacy filtering.')


## Quality Evaluation (TSTR)

In [ ]:
def compute_quality_metrics(X_real, X_synthetic, y_real, y_synthetic, method_name):
    # 1. Statistical similarity (Mean Error)
    mean_real = X_real.mean(axis=0)
    mean_synth = X_synthetic.mean(axis=0)
    mean_error = np.mean(np.abs(mean_real - mean_synth))
    
    # 2. Correlation similarity
    corr_real = np.corrcoef(X_real.T)
    corr_synth = np.corrcoef(X_synthetic.T)
    corr_error = np.mean(np.abs(corr_real - corr_synth))
    
    # 3. Downstream utility (K-NN TSTR)
    knn_synth = KNeighborsClassifier(n_neighbors=3)
    knn_synth.fit(X_synthetic, y_synthetic)
    utility = knn_synth.score(X_real, y_real)
    
    return {
        'Method': method_name,
        'Mean Error': mean_error,
        'Corr Error': corr_error,
        'Utility (TSTR)': utility
    }


In [ ]:
print('\nComputing quality metrics...')
metrics_vae = compute_quality_metrics(X_real, X_synthetic_vae, y_real, y_synthetic_vae, 'CVAE')
metrics_gan = compute_quality_metrics(X_real, X_synthetic_gan, y_real, y_synthetic_gan, 'CGAN')

df_metrics = pd.DataFrame([metrics_vae, metrics_gan])
print(df_metrics)

## PART 3: Membership Inference Attack (MIA) Resistance

In [ ]:
def nearest_neighbor_distance(data_point, dataset):
    distances = np.linalg.norm(dataset - data_point, axis=1)
    return distances.min()

def check_mia_resistance(X_synthetic, X_real, label=""):
    synth_to_real = [nearest_neighbor_distance(s, X_real) for s in X_synthetic]
    
    real_to_real = []
    for i, r in enumerate(X_real):
        dists = np.linalg.norm(X_real - r, axis=1)
        dists[i] = np.inf
        real_to_real.append(dists.min())
        
    s_mean, s_std = np.mean(synth_to_real), np.std(synth_to_real)
    r_mean, r_std = np.mean(real_to_real), np.std(real_to_real)
    
    print(f'\nMembership Inference Analysis ({label}):')
    print(f'  Synthetic -> Nearest Real: {s_mean:.4f} ± {s_std:.4f}')
    print(f'  Real -> Nearest Real: {r_mean:.4f} ± {r_std:.4f}')
    
    if s_mean > r_mean:
        print("  ✓ Synthetic samples are DISTINCT")
        print("  ✓ Hardened against membership inference")
    else:
        print("  ✗ Synthetic samples are SIMILAR (Privacy Risk)")


In [ ]:
check_mia_resistance(X_synthetic_vae, X_real, 'CVAE')
check_mia_resistance(X_synthetic_gan, X_real, 'CGAN')

## Final Visual Verification (PCA)

In [ ]:
pca = PCA(n_components=2)
X_real_pca = pca.fit_transform(X_real_scaled)
X_vae_pca = pca.transform(scaler.transform(X_synthetic_vae))
X_gan_pca = pca.transform(scaler.transform(X_synthetic_gan))

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.scatter(X_real_pca[:, 0], X_real_pca[:, 1], c=y_real, alpha=0.5, cmap='viridis')
plt.title('Real Data (PCA)')

plt.subplot(1, 3, 2)
plt.scatter(X_vae_pca[:, 0], X_vae_pca[:, 1], c=y_synthetic_vae, alpha=0.5, cmap='viridis')
plt.title('CVAE Synthetic (PCA)')

plt.subplot(1, 3, 3)
plt.scatter(X_gan_pca[:, 0], X_gan_pca[:, 1], c=y_synthetic_gan, alpha=0.5, cmap='viridis')
plt.title('CGAN Synthetic (PCA)')

plt.tight_layout()
plt.show()


## Exercises

### Exercise 1: Conditional Synthetic Data
Extend VAE/GAN to generate class-conditional synthetic data:
- Add class label as input to decoder/generator
- Generate synthetic samples for each Iris class separately
- Evaluate if class distributions are preserved

### Exercise 2: Differential Privacy Integration
Implement differentially private training:
- Add noise to gradients during VAE/GAN training
- Measure ε-δ privacy budget
- Compare privacy-utility trade-offs

### Exercise 3: Feature Importance via Synthetic
Use synthetic data to understand feature importance:
- Generate synthetic data with one feature removed/shuffled
- Measure ML model performance degradation
- Compare to SHAP/permutation importance

### Exercise 4: Adversarial Detection
Train adversarial detector to identify synthetic vs real:
- Train classifier on [real data, synthetic data] labels
- Measure AUC (detecting synthetic data)
- If AUC > 0.7, synthetic data quality is poor

### Exercise 5: Multi-Modal Data
Extend to mixed data types:
- Continuous features (using Normal distribution)
- Categorical features (using Categorical distribution)
- Implement on UCI Adult dataset

### Exercise 6: Anonymization Evaluation
Compare approaches for data release:
- Real data (baseline, not private)
- Synthetic data (fully private)
- k-Anonymized real data
- Differentially private real data

Which achieves best privacy-utility balance for ML?